In [ ]:
# --.. ..- .-.. .-.. --- --.. ..- .-.. .-.. --- --.. ..- .-.. .-.. ---
# Z3ST: An open-source FEniCSx framework for thermo-mechanical analysis
# Author: Giovanni Zullo
# Version: 0.4.1 (2026)
# --.. ..- .-.. .-.. --- --.. ..- .-.. .-.. --- --.. ..- .-.. .-.. ---

# Interactive 3D fields

Reads `output/fields.xdmf` through `gui_fields.load_grid`. Stress and strain are in the local frame of the flats: nn through the wall, ss along the flat, zz along the axis. `sigma_vm` is the von Mises stress, frame-independent.

Needs the trame backend: `pip install "pyvista[jupyter]"`.

In [ ]:
import numpy as np
import pyvista as pv
import ipywidgets as widgets

from gui_fields import D, H, XDMF, cells_as_points, load_grid
from z3st.utils.utils_extract_xdmf import list_steps_xdmf

pv.set_jupyter_backend("trame")
N_STEPS = len(list_steps_xdmf(XDMF, "Displacement"))


WARP_FRACTION = 0.2   # auto warp: peak |u| drawn as this fraction of H, enough to see the bow


def show(field="T (K)", step=-1, deformed=True, warp=0.0, cut=True, edges=False, view="iso"):
    """One field on the original or deformed shape; ``warp=0`` draws the peak |u| as
    ``WARP_FRACTION`` of H (never below true scale), ``cut`` keeps the lower half so the
    through-wall gradient shows on the cut face, ``view="side"`` looks along x at the
    y-z plane, where the duct bows."""
    grid = load_grid(step)
    if deformed:
        u_max = max(np.linalg.norm(grid.point_data["u"], axis=1).max(), 1e-30)
        warp = warp or max(1.0, WARP_FRACTION * H / u_max)
        mesh = grid.warp_by_vector("u", factor=warp)
    else:
        mesh = grid
    if cut:
        mesh = mesh.clip(normal="z", origin=(0, 0, H / 2))
    p = pv.Plotter()
    title = f"{field}, step {step}" + (f", deformed x{warp:.3g}" if deformed else "")
    p.add_text(title, font_size=10)
    if field is None:
        p.add_mesh(mesh, color="lightgray", show_edges=True, line_width=0.3)
    else:
        if field in mesh.cell_data:
            mesh = cells_as_points(mesh)
        p.add_mesh(mesh, scalars=field, preference="point", interpolate_before_map=False, cmap="turbo", show_edges=edges,
                   scalar_bar_args={"title": field, "fmt": "%.4g"})
    if deformed:
        p.add_mesh(grid.clip(normal="z", origin=(0, 0, H / 2)) if cut else grid,
                   style="wireframe", color="black", opacity=0.05)
    if view == "side":   # parallel projection: no perspective skew between the two ends
        p.view_yz()
        p.enable_parallel_projection()
    else:
        p.view_isometric()
    p.show()

## Mesh

In [ ]:
show(None, deformed=False, cut=False)

Widget(value='<iframe src="http://localhost:36509/index.html?ui=P_0x7e454d7fc650_0&reconnect=auto" class="pyvi…

## Original shape, temperature

In [ ]:
show("T (K)", deformed=False)

Widget(value='<iframe src="http://localhost:36509/index.html?ui=P_0x7e454d7bc7d0_1&reconnect=auto" class="pyvi…

## Deformed shape, displacement magnitude

In [ ]:
show("u (um)")

Widget(value='<iframe src="http://localhost:36509/index.html?ui=P_0x7e450c5e1a30_2&reconnect=auto" class="pyvi…

## Flexure

Whole length seen from the side (y-z plane): the bow of the duct, amplified.


In [ ]:
show("u (um)", cut=False, view="side")

Widget(value='<iframe src="http://localhost:36509/index.html?ui=P_0x7e454d7eaf90_3&reconnect=auto" class="pyvi…

## Stress

In [ ]:
show("sigma_ss (MPa)")

/home/mat/z3st_macro/z3st/cases/sandbox/U_hex_shell/gui_fields.py:84: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  s = mesh.extract_surface().cast_to_unstructured_grid().separate_cells()


Widget(value='<iframe src="http://localhost:36509/index.html?ui=P_0x7e454de64230_4&reconnect=auto" class="pyvi…

## Von Mises stress

In [ ]:
show("sigma_vm (MPa)")

Widget(value='<iframe src="http://localhost:36509/index.html?ui=P_0x7e454d7bdc40_5&reconnect=auto" class="pyvi…

## Strain

In [ ]:
show("eps_ss (-)")

Widget(value='<iframe src="http://localhost:36509/index.html?ui=P_0x7e450c027200_6&reconnect=auto" class="pyvi…

## Explorer

Any field, step, warp, cut and view. Each change rebuilds the view.


In [ ]:
g = load_grid()
fields = [k for k in g.point_data.keys() if k != "u"] + list(g.cell_data.keys())
widgets.interact(
    show,
    field=widgets.Dropdown(options=sorted(fields), value="sigma_ss (MPa)"),
    step=widgets.IntSlider(min=0, max=N_STEPS - 1, value=N_STEPS - 1),
    deformed=True,
    warp=widgets.FloatText(value=0.0, description="warp (0 = auto)"),
    cut=True,
    edges=False,
    view=widgets.Dropdown(options=["iso", "side"], value="iso"),
);

interactive(children=(Dropdown(description='field', index=5, options=('T (K)', 'eps_nn (-)', 'eps_ss (-)', 'ep…

## Section

The whole cross-section at a chosen height, seen from above: all six flats and the full wall thickness. Uses `fields` from the Explorer cell above.

In [ ]:
def show_section(field="sigma_vm (MPa)", step=-1, z=0.5, deformed=False, warp=0.0, edges=True):
    """Whole hexagonal cross-section at height ``z`` (fraction of H), seen from +z;
    ``warp=0`` draws the section's peak in-plane |u| as 5 % of D, never below true scale."""
    z_cut = np.clip(z * H, 1e-6 * H, (1 - 1e-6) * H)   # inside the mesh at the ends
    sec = load_grid(step).slice(normal="z", origin=(0, 0, z_cut))
    if deformed:   # cut first, then warp: the section stays the one at z_cut
        u_xy = np.linalg.norm(sec.point_data["u"][:, :2], axis=1).max()
        warp = warp or max(1.0, 0.05 * D / max(u_xy, 1e-30))
        sec = sec.warp_by_vector("u", factor=warp)
    if field in sec.cell_data:
        sec = cells_as_points(sec)
    p = pv.Plotter()
    title = f"{field}, step {step}, z = {z_cut:.3f} m" + (f", deformed x{warp:.3g}" if deformed else "")
    p.add_text(title, font_size=10)
    p.add_mesh(sec, scalars=field, preference="point", interpolate_before_map=False, cmap="turbo",
               show_edges=edges, line_width=0.3, scalar_bar_args={"title": field, "fmt": "%.4g"})
    p.view_xy()
    p.show()


widgets.interact(
    show_section,
    field=widgets.Dropdown(options=sorted(fields), value="sigma_vm (MPa)"),
    step=widgets.IntSlider(min=0, max=N_STEPS - 1, value=N_STEPS - 1),
    z=widgets.FloatSlider(min=0.0, max=1.0, step=1 / 40, value=0.5, description="z / H"),
    deformed=False,
    warp=widgets.FloatText(value=0.0, description="warp (0 = auto)"),
    edges=True,
);

interactive(children=(Dropdown(description='field', index=6, options=('T (K)', 'eps_nn (-)', 'eps_ss (-)', 'ep…